In [7]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

import pandas as pd
import requests

In [8]:
FAMILY_MAP = {
    "Aire acondicionado": "AA",
    "Estacion de Carga Portatil": "GENKI",
    "TPMS": "TPMS",
    "TPMS Repuesto": "TPMS",
    "Climatizador": "CLIMATIZADOR",
    "Carjack": "CARJACK",
    "Arrancador": "CARJACK",
    "Inflador": "CARJACK",
    "Caldera": "CALDERA",
    "Solar": "SOLAR",
}

In [9]:
def normalize_sku(x) -> str:
    return str(x).strip()


def split_images(s: str) -> list[str]:
    if not isinstance(s, str) or not s.strip():
        return []
    return [u.strip() for u in s.split(",") if u.strip()]


def to_bool(x):
    if pd.isna(x):
        return None
    return bool(x)

In [16]:
def export_from_csv(csv_path_or_url: str, out_path: Path):
    if csv_path_or_url.startswith("http"):
        response = requests.get(csv_path_or_url)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
    else:
        df = pd.read_csv(csv_path_or_url)

    items = []

    for _, r in df.iterrows():
        title   = str(r["title"]).strip() if pd.notna(r["title"]) else ""
        product_family = r["product_family"] if pd.notna(r["product_family"]) else ""
        # Determine family from or title
        fam = "OTHER"
        for key in FAMILY_MAP:
            if key.lower() in product_family.lower() or key.lower() in title.lower():
                fam = FAMILY_MAP[key]
                break

        sku     = normalize_sku(r["id"])
        model   = str(r["model"]).strip() if pd.notna(r["model"]) else ""
        
        # Parse price: remove " ARS", then handle Argentine format (dots as thousands, comma as decimal)
        if pd.notna(r["sale_price"]):
            price_str = str(r["sale_price"]).replace(" ARS", "").strip()
            price_str = price_str.replace(".", "").replace(",", ".")
            try:
                price = float(price_str)
            except ValueError:
                price = None
        else:
            price = None
        
        sales_link = str(r["link"]).strip() if pd.notna(r["link"]) else None


        items.append({
            "product_family": fam, 
            "sku": sku,
            "model": model,
            "title": title,
            "price": price,
            "sales_link": sales_link,
        })

    catalog = {
        "schema_version": "catalog.v1",
        "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "defaults": {"currency": "ARS", "country": "AR"},
        "source": {
            "type": "csv",
            "file": csv_path_or_url,
        },
        "items": sorted(items, key=lambda x: x["sku"]),
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(
        catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Wrote {len(catalog['items'])} items to {out_path}")

In [17]:
# Path or URL to CSV source
in_path = "../knowledge/facebook-feed.csv"  # "https://neil.ar/facebook-feed.csv"
out_path = Path("../data/catalog.json")  # Output catalog.json path

if in_path.endswith(".csv") or in_path.startswith("http"):
    export_from_csv(in_path, out_path)
else:
    raise SystemExit("Unsupported input type. Use .csv or URL")

Wrote 57 items to ../data/catalog.json
